# Analisis Completo de Baterias 13300

Procesamiento y analisis de datos de los 36 baterias de litio extraidas de vapeadores desechables.

Este cuaderno realiza todas las operaciones en secuencia:
1. Organizacion de archivos de cicladores por fecha
2. Extraccion de datos de SoH (State of Health)
3. Extraccion de impedancia mediante EIS
4. Seleccion de 18 baterias compatibles
5. Exportacion de resultados a CSV

## Imports y Configuracion

In [ ]:
"""
IMPORTS

Import all necessary modules from the project package structure.
"""

# System imports
from pathlib import Path

# Data processing
import numpy as np
import pandas as pd

# Project imports
from src.data_io.file_organization import organizar_todos_los_cicladores
from src.processing.cycling import trim_txt_files, build_soh_matrix, load_batteries_from_csv
from src.processing.eis import build_zcurve_matrix, actualizar_impedancia
from src.analysis.soh import find_best_18
from src.data_io.export import export_best_18_batteries
from config import CYCLER_RANGE, OUTPUT_DIR

print("OK: All modules imported successfully!")

OK: All modules imported successfully!


## Paso 1: Organizacion de Archivos por Fecha

Organiza los archivos brutos de los cicladores (ciclador_1 a ciclador_6) en una estructura ordenada por fecha.
Los archivos originales se preservan sin modificacion.

In [ ]:
print("STEP 1: Organizing raw cycler files by date...")
print("="*60)

results_org = organizar_todos_los_cicladores()

print("\nOrganization complete!")
print("="*60)

STEP 1: Organizing raw cycler files by date...
Organizando archivos de todos los cicladores por fecha...

Procesando: data/raw/cicladores/ciclador_1 ...
  OK Organizados: 46
  SKIP Omitidos: 0

Procesando: data/raw/cicladores/ciclador_2 ...
  OK Organizados: 58
  SKIP Omitidos: 0

Procesando: data/raw/cicladores/ciclador_3 ...
  OK Organizados: 52
  SKIP Omitidos: 0

Procesando: data/raw/cicladores/ciclador_4 ...
  OK Organizados: 48
  SKIP Omitidos: 0

Procesando: data/raw/cicladores/ciclador_5 ...
  OK Organizados: 50
  SKIP Omitidos: 0

Procesando: data/raw/cicladores/ciclador_6 ...
  OK Organizados: 46
  SKIP Omitidos: 0

OK: Todas las carpetas han sido organizadas!

Resumen total:
  Organizados: 300
  Omitidos: 0

Organization complete!


## Paso 2: Procesamiento de Archivos de Ciclado

Procesa los archivos de ciclado para mantener solo:
- Fila de encabezado (nombres de columnas)
- Ultima fila de datos (estado final del ciclo)

Esto reduce el tamanio del archivo y enfoca el analisis en el estado final de cada bateria.

In [ ]:
print("STEP 2: Trimming cycling files...")
print("="*60)

results_trim = trim_txt_files()

print(f"\nTrimming complete!")
print("="*60)

STEP 2: Trimming cycling files...
OK Trimmed : 129 .txt files
SKIP Skipped : 21 .txt files (already 2 lines or empty)
DELETE Deleted : 150 .dat files
ERROR Errors  : 0 files

Trimming complete!


## Paso 3: Extraccion de SoH (State of Health)

Extrae el valor final de SoH de cada bateria de sus archivos procesados.
Crea una matriz CSV con los numeros de bateria, valores de SoH e impedancia (a completar en el siguiente paso).

In [ ]:
print("STEP 3: Building SoH (State of Health) matrix...")
print("="*60)

baterias = build_soh_matrix()

print(f"\nOK: Built SoH matrix with {len(baterias)} batteries")
print("="*60)

STEP 3: Building SoH (State of Health) matrix...

OK: SoH matrix saved to: '/home/elias/Documents/TFG/Procesado_Datos/TFG_Elias/data/processed/matriz_baterias.csv'

OK: Built SoH matrix with 36 batteries


## Paso 4: Extraccion de Impedancia mediante EIS

Extrae los datos de Espectroscopia de Impedancia Electroqu"mica (EIS) de los archivos .DTA.
Construye una matriz ZCURVE con valores de impedancia real e imaginaria.

In [ ]:
print("STEP 4: Building ZCURVE impedance matrix from EIS files...")
print("="*60)

df_zcurve = build_zcurve_matrix()

if df_zcurve is not None:
    print(f"\nZCURVE matrix statistics:")
    print(f"  Shape: {df_zcurve.shape}")
    print(f"  Columns: {list(df_zcurve.columns)}")
    print(f"  Batteries detected: {df_zcurve['Bateria_Num'].nunique()}")
    print(f"\nFirst few rows:")
    print(df_zcurve.head())
else:
    print("Warning: Could not build ZCURVE matrix")

print("="*60)

STEP 4: Building ZCURVE impedance matrix from EIS files...
OK: ZCURVE matrix saved to: '/home/elias/Documents/TFG/Procesado_Datos/TFG_Elias/data/processed/matriz_zcurve.csv'

ZCURVE matrix statistics:
  Shape: (1836, 15)
  Columns: ['Bateria_Num', 'Pt', 'Time', 'Freq', 'Zreal', 'Zimag', 'Zsig', 'Zmod', 'Zphz', 'Idc', 'Vdc', 'IERange', 'Imod', 'Vmod', 'Temp']
  Batteries detected: 36

First few rows:
   Bateria_Num  Pt  Time       Freq     Zreal     Zimag  Zsig      Zmod  \
0            1   0     5  10019.530  0.086353 -0.002874     1  0.086401   
1            1   1     6   7928.241  0.087350 -0.004214     1  0.087451   
2            1   2     8   6307.871  0.088417 -0.005411     1  0.088583   
3            1   3     9   5034.722  0.089539 -0.006484     1  0.089774   
4            1   4    11   3993.056  0.090809 -0.007459     1  0.091115   

       Zphz       Idc       Vdc  IERange      Imod      Vmod      Temp  
0 -1.906296 -0.000037  4.039976       11  0.049518  0.004278  1473.314  


## Paso 5: Actualizacion de Matriz de Baterias con Impedancia

Usa los datos ZCURVE para encontrar el valor de impedancia en el punto de cruce cero de cada bateria
(donde Zimag esta mas cercano a cero en la grafica de Nyquist).
Esto representa la resistencia de transferencia de carga e impedancia total.

In [ ]:
print("STEP 5: Updating battery matrix with impedance values...")
print("="*60)

df_baterias_updated = actualizar_impedancia()

if df_baterias_updated is not None:
    print(f"\nOK: Battery matrix updated with impedance values!")
    print(f"\nUpdated battery matrix (first 10 rows):")
    print(df_baterias_updated.head(10))
else:
    print("Warning: Could not update battery matrix")

print("="*60)

STEP 5: Updating battery matrix with impedance values...
Loading data...
ERROR: 'Impedancia'


## Paso 6: Seleccion de 18 Baterias Compatibles

Encuentra las 18 baterias con los valores de SoH mas similares.
Esto asegura una distribucion uniforme de carga y rendimiento equilibrado en el pack final.

Algoritmo: Encuentra una ventana deslizante de 18 baterias (ordenadas por SoH) que minimiza el rango.

In [ ]:
print("STEP 6: Selecting 18 best compatible batteries...")
print("="*60)

best_baterias = [b for b in find_best_18(baterias) if b is not None]

print(f"\nOK: Selected {len(best_baterias)} compatible batteries:")
print(f"\nBattery Numbers: {sorted([b.numero for b in best_baterias])}")

# Calculate statistics
soh_values = [b.soh for b in best_baterias]
impedance_values = [b.impedancia for b in best_baterias]

print(f"\nSOH Statistics:")
print(f"  Mean: {np.mean(soh_values):.4f} Ah")
print(f"  Std Dev: {np.std(soh_values):.4f} Ah")
print(f"  Min: {np.min(soh_values):.4f} Ah")
print(f"  Max: {np.max(soh_values):.4f} Ah")
print(f"  Range: {np.max(soh_values) - np.min(soh_values):.4f} Ah")

print(f"\nImpedance Statistics:")
print(f"  Mean: {np.mean(impedance_values):.4f} Ohm")
print(f"  Std Dev: {np.std(impedance_values):.4f} Ohm")
print(f"  Min: {np.min(impedance_values):.4f} Ohm")
print(f"  Max: {np.max(impedance_values):.4f} Ohm")

print("="*60)

STEP 6: Selecting 18 best compatible batteries...

OK: Selected 18 compatible batteries:

Battery Numbers: [1, 2, 6, 7, 9, 10, 12, 13, 15, 17, 20, 21, 24, 26, 29, 33, 35, 36]

SOH Statistics:
  Mean: 0.3402 Ah
  Std Dev: 0.0041 Ah
  Min: 0.3332 Ah
  Max: 0.3463 Ah
  Range: 0.0131 Ah

Impedance Statistics:
  Mean: 0.0000 Ohm
  Std Dev: 0.0000 Ohm
  Min: 0.0000 Ohm
  Max: 0.0000 Ohm


## Paso 7: Informacion Detallada del Pack Seleccionado

In [ ]:
print("\n" + "="*70)
print("SELECTED BATTERY PACK - DETAILED INFORMATION")
print("="*70)

for i, bat in enumerate(sorted(best_baterias, key=lambda b: b.numero), 1):
    print(f"{i:2d}. {bat}")

print("\n" + "="*70)


SELECTED BATTERY PACK - DETAILED INFORMATION
 1. Batería #1 | SoH: 0.335735Ah | Impedancia: 0Ω
 2. Batería #2 | SoH: 0.340149Ah | Impedancia: 0Ω
 3. Batería #6 | SoH: 0.345978Ah | Impedancia: 0Ω
 4. Batería #7 | SoH: 0.339422Ah | Impedancia: 0Ω
 5. Batería #9 | SoH: 0.342146Ah | Impedancia: 0Ω
 6. Batería #10 | SoH: 0.340374Ah | Impedancia: 0Ω
 7. Batería #12 | SoH: 0.335763Ah | Impedancia: 0Ω
 8. Batería #13 | SoH: 0.343665Ah | Impedancia: 0Ω
 9. Batería #15 | SoH: 0.346292Ah | Impedancia: 0Ω
10. Batería #17 | SoH: 0.333173Ah | Impedancia: 0Ω
11. Batería #20 | SoH: 0.344466Ah | Impedancia: 0Ω
12. Batería #21 | SoH: 0.345323Ah | Impedancia: 0Ω
13. Batería #24 | SoH: 0.334309Ah | Impedancia: 0Ω
14. Batería #26 | SoH: 0.344176Ah | Impedancia: 0Ω
15. Batería #29 | SoH: 0.337994Ah | Impedancia: 0Ω
16. Batería #33 | SoH: 0.334904Ah | Impedancia: 0Ω
17. Batería #35 | SoH: 0.340035Ah | Impedancia: 0Ω
18. Batería #36 | SoH: 0.339868Ah | Impedancia: 0Ω



## Paso 8: Exportacion de Resultados

Exporta las 18 baterias seleccionadas a un archivo CSV en el directorio data/output/.
El archivo contiene los numeros de bateria, valores de SoH e impedancia.

In [ ]:
print("STEP 8: Exporting best 18 batteries to CSV...")
print("="*60)

output_filepath = export_best_18_batteries(best_baterias)

print(f"\nOK: Best 18 batteries exported to:")
print(f"    {output_filepath}")

# Display the exported data
exported_df = pd.read_csv(output_filepath)
print(f"\nExported data (sorted by battery number):")
exported_df = exported_df.sort_values('numero').reset_index(drop=True)
print(exported_df.to_string(index=False))

print("\n" + "="*60)
print("ANALYSIS COMPLETE - All results exported successfully!")
print("="*60)

STEP 8: Exporting best 18 batteries to CSV...

OK: Best 18 batteries exported to:
    /home/elias/Documents/TFG/Procesado_Datos/TFG_Elias/data/output/mejores_18_baterias.csv

Exported data (sorted by battery number):
 numero      soh  impedancia
      1 0.335735           0
      2 0.340149           0
      6 0.345978           0
      7 0.339422           0
      9 0.342146           0
     10 0.340374           0
     12 0.335763           0
     13 0.343665           0
     15 0.346292           0
     17 0.333173           0
     20 0.344466           0
     21 0.345323           0
     24 0.334309           0
     26 0.344176           0
     29 0.337994           0
     33 0.334904           0
     35 0.340035           0
     36 0.339868           0

ANALYSIS COMPLETE - All results exported successfully!
